# TP11 — Optuna GroupKFold

B8 (TP8b) a été optimisé sur le **split temporel** — un critère biaisé (mêmes machines en train et val).  
B11 re-tune les hyperparamètres avec **GroupKFold(n_splits=5)** comme objectif Optuna → optimisation honnête.

| Modèle | Optuna objectif | PR-AUC CV (15 folds) | PR-AUC test |
|--------|----------------|----------------------|-------------|
| B9-GKF (TP9) | HP de B8, évalué en GroupKFold(15) | 0.8669 ± 0.0969 | 0.8948 |
| **B11-GKF** | GroupKFold(5) | ? | ? |

## 1. Imports

In [ ]:
import sys
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

from sqlalchemy import create_engine
from sqlalchemy.engine import URL
from sklearn.impute import SimpleImputer
from sklearn.model_selection import GroupKFold
from sklearn.pipeline import Pipeline
from sklearn.metrics import average_precision_score, roc_auc_score, f1_score
from xgboost import XGBClassifier

sys.path.insert(0, str(Path("src")))
from indusense.modeling.dataset import load_gold_dataset
from indusense.modeling.pipeline import build_xgb_pipeline

RANDOM_STATE = 42
N_TRIALS     = 30
N_FOLDS_OPT  = 5   # folds dans l'objectif Optuna (rapidité)
N_FOLDS_EVAL = 15  # folds pour l'évaluation finale (toutes les machines)
np.random.seed(RANDOM_STATE)
print("Imports OK")

## 2. Données

In [ ]:
url = URL.create(
    drivername="postgresql+psycopg2",
    username="indusense_user",
    password="ThEP@ssW0rd",
    host="localhost", port=5432, database="indusense_db",
)
engine = create_engine(url)
gold = load_gold_dataset(engine)

df           = gold.df
FEATURE_COLS = gold.feature_cols
X_tv, y_tv   = gold.X_tv, gold.y_tv
groups       = gold.groups
X_test, y_test = gold.X_test, gold.y_test

SPW_GLOBAL = round((y_tv == 0).sum() / (y_tv == 1).sum(), 2)
N_MACHINES = groups.nunique()

print(f"Features : {len(FEATURE_COLS)} | Machines : {N_MACHINES}")
print(f"Train+val : {len(X_tv):,} | Test : {len(X_test):,}")
print(f"scale_pos_weight global : {SPW_GLOBAL}")

## 3. Objectif Optuna — GroupKFold(5)

Chaque trial Optuna entraîne 5 modèles (un par fold) et retourne la moyenne des PR-AUC.  
L'optimisation cible directement la généralisation machine.

In [ ]:
gkf_opt = GroupKFold(n_splits=N_FOLDS_OPT)

def objective(trial):
    params = {
        "n_estimators":      trial.suggest_int("n_estimators",      50, 600),
        "max_depth":         trial.suggest_int("max_depth",          3, 10),
        "learning_rate":     trial.suggest_float("learning_rate",    0.01, 0.3, log=True),
        "subsample":         trial.suggest_float("subsample",        0.5, 1.0),
        "colsample_bytree":  trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "min_child_weight":  trial.suggest_int("min_child_weight",   1, 20),
        "reg_alpha":         trial.suggest_float("reg_alpha",        1e-4, 10.0, log=True),
        "reg_lambda":        trial.suggest_float("reg_lambda",       1e-4, 10.0, log=True),
        "scale_pos_weight":  SPW_GLOBAL,
        "random_state":      RANDOM_STATE,
        "verbosity":         0,
    }

    scores = []
    for tr_idx, vl_idx in gkf_opt.split(X_tv, y_tv, groups):
        X_tr, y_tr = X_tv.iloc[tr_idx], y_tv.iloc[tr_idx]
        X_vl, y_vl = X_tv.iloc[vl_idx], y_tv.iloc[vl_idx]

        pipe = build_xgb_pipeline(params)
        pipe.fit(X_tr, y_tr)

        n_pos = int(y_vl.sum())
        if n_pos < 2:
            continue
        y_prob = pipe.predict_proba(X_vl)[:, 1]
        scores.append(average_precision_score(y_vl, y_prob))

    return np.mean(scores) if scores else 0.0


print(f"Lancement Optuna — {N_TRIALS} trials × {N_FOLDS_OPT} folds = {N_TRIALS * N_FOLDS_OPT} fits")
print("Patience...")

study = optuna.create_study(direction="maximize",
                            sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE))
study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=True)

best = study.best_params
best["scale_pos_weight"] = SPW_GLOBAL
best["random_state"]     = RANDOM_STATE
best["verbosity"]        = 0

print(f"\nMeilleur PR-AUC CV(5) : {study.best_value:.4f}")
print("Meilleurs params :")
for k, v in best.items():
    print(f"  {k:<22} : {v}")

## 4. Comparaison params B8 vs B11

In [4]:
B8_PARAMS = {
    "n_estimators": 289, "max_depth": 4, "learning_rate": 0.12806863970210308,
    "subsample": 0.7206901602171134, "colsample_bytree": 0.8293762849110333,
    "min_child_weight": 1, "reg_alpha": 0.35765555948723726, "reg_lambda": 0.5629609134253009,
}
HP_KEYS = ["n_estimators", "max_depth", "learning_rate", "subsample",
           "colsample_bytree", "min_child_weight", "reg_alpha", "reg_lambda"]

print(f"{'HP':<22} {'B8 (split temporel)':>22} {'B11 (GroupKFold)':>18}")
print("-" * 66)
for k in HP_KEYS:
    b8v  = B8_PARAMS.get(k, "—")
    b11v = best.get(k, "—")
    diff = " ←" if str(b8v) != str(b11v) else ""
    print(f"  {k:<20} {str(b8v):>22} {str(b11v):>18}{diff}")

HP                        B8 (split temporel)   B11 (GroupKFold)
------------------------------------------------------------------
  n_estimators                            289                502 ←
  max_depth                                 4                  5 ←
  learning_rate           0.12806863970210308 0.032709132548852674 ←
  subsample                0.7206901602171134 0.8098203720913801 ←
  colsample_bytree         0.8293762849110333 0.8219637068265245 ←
  min_child_weight                          1                  3 ←
  reg_alpha               0.35765555948723726 0.0007073976652217782 ←
  reg_lambda               0.5629609134253009 0.0020802438153048933 ←


## 5. Évaluation B11 — GroupKFold(15 machines)

In [ ]:
gkf_eval = GroupKFold(n_splits=N_MACHINES)
fold_results = []

for fold_i, (tr_idx, vl_idx) in enumerate(gkf_eval.split(X_tv, y_tv, groups)):
    machine_val = groups.iloc[vl_idx].unique()[0]
    X_tr, y_tr = X_tv.iloc[tr_idx], y_tv.iloc[tr_idx]
    X_vl, y_vl = X_tv.iloc[vl_idx], y_tv.iloc[vl_idx]

    spw = round((y_tr == 0).sum() / max((y_tr == 1).sum(), 1), 2)
    params = {**best, "scale_pos_weight": spw}
    pipe = build_xgb_pipeline(params)
    pipe.fit(X_tr, y_tr)

    n_pos  = int(y_vl.sum())
    y_prob = pipe.predict_proba(X_vl)[:, 1]
    pr_auc = average_precision_score(y_vl, y_prob) if n_pos >= 2 else float("nan")

    fold_results.append({"machine": machine_val, "n_pos": n_pos, "pr_auc": round(pr_auc, 4)})
    print(f"Fold {fold_i+1:2d} | {machine_val:<10} | n_pos={n_pos:4d} | PR-AUC={pr_auc:.4f}")

results_df = pd.DataFrame(fold_results)
valid = results_df.dropna(subset=["pr_auc"])

# B9-GKF (TP9) — PR-AUC par machine, run rejoué avec le Gold Dataset + capacity/max/future_incident_count, sans fuite
B9_SCORES = {
    "MACH-15": 0.9644, "MACH-01": 0.9206, "MACH-12": 0.9394, "MACH-07": 0.9473,
    "MACH-05": 0.5921, "MACH-11": 0.9988, "MACH-10": 0.9171, "MACH-14": 0.9136,
    "MACH-13": 0.7743, "MACH-04": 0.8633, "MACH-02": 0.9193, "MACH-06": 0.9613,
    "MACH-08": 0.9902, "MACH-03": 0.6868, "MACH-09": 0.7806,
}
valid = valid.copy()
valid["b9"] = valid["machine"].map(B9_SCORES)
valid["delta"] = valid["pr_auc"] - valid["b9"]

print(f"\nPR-AUC CV moyen : {valid['pr_auc'].mean():.4f} ± {valid['pr_auc'].std():.4f}")
print(f"Δ vs B9-GKF     : {valid['pr_auc'].mean() - 0.8779:+.4f}")

## 6. Modèle final — test

In [ ]:
pipe_final = build_xgb_pipeline(best)
pipe_final.fit(X_tv, y_tv)

pr_train = average_precision_score(y_tv,   pipe_final.predict_proba(X_tv)[:, 1])
pr_test  = average_precision_score(y_test, pipe_final.predict_proba(X_test)[:, 1])
roc_test = roc_auc_score(y_test,           pipe_final.predict_proba(X_test)[:, 1])
f1_test  = f1_score(y_test,                pipe_final.predict(X_test), zero_division=0)
delta_tv = round(pr_train - valid["pr_auc"].mean(), 4)

print("=== Modèle final B11-GKF ===")
print(f"  PR-AUC train    : {pr_train:.4f}")
print(f"  PR-AUC CV moyen : {valid['pr_auc'].mean():.4f} ± {valid['pr_auc'].std():.4f}")
print(f"  PR-AUC test     : {pr_test:.4f}")
print(f"  ROC-AUC test    : {roc_test:.4f}")
print(f"  F1 test         : {f1_test:.4f}")
print(f"  Overfitting Δ   : {delta_tv:.4f}")

print("\n=== Comparaison globale ===")
print(f"  {'Modèle':<14} {'Optuna obj.':<20} {'PR-AUC CV':>10} {'PR-AUC test':>13} {'F1':>8}")
print("  " + "-"*70)
print(f"  {'B9-GKF (TP9)':<14} {'HP de B8':<20} {'0.8779':>10} {'0.8709':>13} {'0.7908':>8}")
print(f"  {'B11-GKF':<14} {'GroupKFold(5)':<20} {valid['pr_auc'].mean():>10.4f} {pr_test:>13.4f} {f1_test:>8.4f}")